# MLB Attendance Analytics

This notebook prepares MLB game-level attendance data, merges it with payroll and performance information, and builds presentation-ready visuals focused on the Cleveland Guardians.

# Project Overview

## Business question
Which factors most influence MLB game attendance, and how can those insights help the Cleveland Guardians improve turnout and ticket revenue?

## Data sources
- **Retrosheet game logs:** home attendance, dates, opponents, and outcomes at the game level
- **Spotrac payroll data:** team payroll by season
- **Baseball Reference team performance:** wins, losses, and win percentage by season
- **Stadium capacity reference table:** used to standardize attendance into `% of capacity`

## What this notebook delivers
- Cleans and standardizes raw attendance and payroll data
- Merges team-level and game-level datasets on `team_abbr` and `season`
- Engineers features such as `attendance_pct`, `weekday`, `month`, and `opponent_payroll_tier`
- Produces presentation-ready visuals that connect timing, opponent quality, payroll, and team performance

## 1. Load Retrosheet Game Logs

Retrosheet game logs are raw TXT files with no header row. This step extracts the fields needed for attendance analysis, cleans dates and numeric fields, and creates time-based features such as month, weekday, and weekend game.

In [18]:
import io
import math
import os
import time
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
import statsmodels.formula.api as smf
from IPython.display import display


RETROSHEET_FILES = {
    2021: "gl2021.txt",
    2022: "gl2022.txt",
    2023: "gl2023.txt",
    2024: "gl2024.txt",
    2025: "gl2025.txt",
}

COLUMN_MAP = {
    0: "date",
    3: "away_team",
    6: "home_team",
    9: "away_score",
    10: "home_score",
    11: "game_length_outs",
    12: "day_night",
    16: "park_id",
    17: "attendance",
}

TEAM_NAME_MAP = {
    "ARI": "Arizona Diamondbacks",
    "ATL": "Atlanta Braves",
    "BAL": "Baltimore Orioles",
    "BOS": "Boston Red Sox",
    "CHN": "Chicago Cubs",
    "CHA": "Chicago White Sox",
    "CIN": "Cincinnati Reds",
    "CLE": "Cleveland Guardians",
    "COL": "Colorado Rockies",
    "DET": "Detroit Tigers",
    "HOU": "Houston Astros",
    "KCA": "Kansas City Royals",
    "ANA": "Los Angeles Angels",
    "LAN": "Los Angeles Dodgers",
    "MIA": "Miami Marlins",
    "MIL": "Milwaukee Brewers",
    "MIN": "Minnesota Twins",
    "NYN": "New York Mets",
    "NYA": "New York Yankees",
    "ATH": "Athletics",
    "PHI": "Philadelphia Phillies",
    "PIT": "Pittsburgh Pirates",
    "SDN": "San Diego Padres",
    "SFN": "San Francisco Giants",
    "SEA": "Seattle Mariners",
    "SLN": "St. Louis Cardinals",
    "TBA": "Tampa Bay Rays",
    "TEX": "Texas Rangers",
    "TOR": "Toronto Blue Jays",
    "WAS": "Washington Nationals",
}

MONTH_ORDER = ["April", "May", "June", "July", "August", "September", "October"]
WEEKDAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
TIER_ORDER = ["Low", "Mid-Low", "Mid-High", "High"]

GUARDIANS_BLUE = "#00385D"
GUARDIANS_RED = "#E31937"
TEXT_COLOR = "#26364D"
GRID_COLOR = "#E8EDF4"
CACHE_PAYROLL_CSV = Path("mlb_team_payroll_2021_2025.csv")


def load_retrosheet_file(filepath):
    """Load one Retrosheet game log and keep the fields needed for attendance analysis."""
    df = pd.read_csv(filepath, header=None, low_memory=False)
    df = df[list(COLUMN_MAP.keys())].rename(columns=COLUMN_MAP).copy()

    df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d", errors="coerce")
    numeric_cols = ["attendance", "away_score", "home_score", "game_length_outs"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["season"] = df["date"].dt.year
    df["home_result"] = np.where(df["home_score"] > df["away_score"], "W", "L")
    df["weekday"] = df["date"].dt.day_name()
    df["month"] = df["date"].dt.month_name()
    df["weekend_game"] = df["weekday"].isin(["Friday", "Saturday", "Sunday"]).astype(int)
    df["home_team_name"] = df["home_team"].map(TEAM_NAME_MAP)
    df["away_team_name"] = df["away_team"].map(TEAM_NAME_MAP)

    return df.dropna(subset=["date", "attendance", "home_team", "away_team"]).copy()


game_frames = []

for season, filename in RETROSHEET_FILES.items():
    if not Path(filename).exists():
        raise FileNotFoundError(
            f"Missing {filename}. Upload the Retrosheet game logs to the notebook working directory before running this cell."
        )
    print(f"Loading {filename}...")
    temp = load_retrosheet_file(filename)
    temp["season"] = season
    game_frames.append(temp)

mlb_games = (
    pd.concat(game_frames, ignore_index=True)
    .sort_values(["season", "date"])
    .reset_index(drop=True)
)

mlb_home_games = mlb_games.rename(
    columns={
        "date": "game_date",
        "home_team": "team_abbr",
        "home_team_name": "team",
        "away_team": "opponent_abbr",
        "away_team_name": "opponent",
        "home_score": "team_score",
        "away_score": "opponent_score",
        "home_result": "result",
    }
)[
    [
        "game_date",
        "season",
        "team_abbr",
        "team",
        "opponent_abbr",
        "opponent",
        "result",
        "attendance",
        "weekday",
        "month",
        "weekend_game",
        "day_night",
        "park_id",
    ]
].copy()

guardians_home_games = (
    mlb_home_games[mlb_home_games["team_abbr"] == "CLE"]
    .sort_values(["season", "game_date"])
    .reset_index(drop=True)
)

print("MLB home games:", mlb_home_games.shape)
print("Guardians home games:", guardians_home_games.shape)
display(mlb_home_games.head())

Loading gl2021.txt...
Loading gl2022.txt...
Loading gl2023.txt...
Loading gl2024.txt...
Loading gl2025.txt...
MLB home games: (12143, 13)
Guardians home games: (403, 13)


,game_date,season,team_abbr,team,opponent_abbr,opponent,result,attendance,weekday,month,weekend_game,day_night,park_id
0,2021-04-01,2021,CHN,Chicago Cubs,PIT,Pittsburgh Pirates,L,10343.0,Thursday,April,0,D,CHI11
1,2021-04-01,2021,CIN,Cincinnati Reds,SLN,St. Louis Cardinals,L,12264.0,Thursday,April,0,D,CIN09
2,2021-04-01,2021,COL,Colorado Rockies,LAN,Los Angeles Dodgers,W,20570.0,Thursday,April,0,D,DEN02
3,2021-04-01,2021,MIA,Miami Marlins,TBA,Tampa Bay Rays,L,7062.0,Thursday,April,0,D,MIA02
4,2021-04-01,2021,MIL,Milwaukee Brewers,MIN,Minnesota Twins,W,11740.0,Thursday,April,0,D,MIL06


## 2. Extract Payroll Data

Spotrac payroll data is scraped by season, then team abbreviations are mapped into the Retrosheet abbreviation format so the payroll table can be merged into the game-level dataset.

In [19]:
SPOTRAC_TO_RETROSHEET = {
    "ARI": "ARI", "ATL": "ATL", "BAL": "BAL", "BOS": "BOS", "CHC": "CHN",
    "CHW": "CHA", "CIN": "CIN", "CLE": "CLE", "COL": "COL", "DET": "DET",
    "HOU": "HOU", "KC": "KCA", "LAA": "ANA", "LAD": "LAN", "MIA": "MIA",
    "MIL": "MIL", "MIN": "MIN", "NYM": "NYN", "NYY": "NYA", "ATH": "ATH",
    "PHI": "PHI", "PIT": "PIT", "SD": "SDN", "SF": "SFN", "SEA": "SEA",
    "STL": "SLN", "TB": "TBA", "TEX": "TEX", "TOR": "TOR", "WSH": "WAS",
}


def scrape_spotrac_cash_payroll(year):
    """Scrape MLB cash payroll by team and convert it into a clean team-season table."""
    url = f"https://www.spotrac.com/mlb/cash/_/year/{year}"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    response.raise_for_status()

    tables = pd.read_html(StringIO(response.text))
    payroll_table = None
    for table in tables:
        cols = [str(col) for col in table.columns]
        if "Team" in cols and any("Total Cash" in col for col in cols):
            payroll_table = table.copy()
            break

    if payroll_table is None:
        raise ValueError(f"Could not find payroll table for {year}")

    payroll_table.columns = [str(col).strip() for col in payroll_table.columns]
    total_cash_col = [col for col in payroll_table.columns if "Total Cash" in col][0]

    payroll_table = payroll_table[["Team", total_cash_col]].rename(
        columns={total_cash_col: "payroll"}
    )
    payroll_table["spotrac_team_abbr"] = (
        payroll_table["Team"].astype(str).str.extract(r"([A-Z]{2,3})", expand=False)
    )
    payroll_table["team_abbr"] = payroll_table["spotrac_team_abbr"].map(SPOTRAC_TO_RETROSHEET)
    payroll_table["payroll"] = (
        payroll_table["payroll"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )
    payroll_table["season"] = year

    return (
        payroll_table[["season", "team_abbr", "payroll"]]
        .dropna()
        .drop_duplicates(subset=["season", "team_abbr"])
        .copy()
    )


payroll_frames = []
if CACHE_PAYROLL_CSV.exists():
    print(f"Loading cached payroll file: {CACHE_PAYROLL_CSV}")
    payroll_df = pd.read_csv(CACHE_PAYROLL_CSV)
else:
    for year in range(2021, 2026):
        print(f"Scraping payroll for {year}...")
        payroll_frames.append(scrape_spotrac_cash_payroll(year))
        time.sleep(2)

    payroll_df = (
        pd.concat(payroll_frames, ignore_index=True)
        .sort_values(["season", "team_abbr"])
        .reset_index(drop=True)
    )
    payroll_df.to_csv(CACHE_PAYROLL_CSV, index=False)

display(payroll_df.head())

Loading cached payroll file: mlb_team_payroll_2021_2025.csv


,season,team_abbr,payroll
0,2021,ANA,196428336
1,2021,ARI,105079998
2,2021,ATL,165565236
3,2021,BAL,64117070
4,2021,BOS,207947136


## 3. Transform and Merge Datasets

This section adds stadium capacity, joins team and opponent payroll, creates attendance percentage, and builds the final team-season summary table used in the analysis.

In [20]:
stadium_capacity = pd.DataFrame({
    "team_abbr": [
        "CLE", "NYA", "LAN", "CHN", "BOS", "ATL", "HOU", "SFN", "SEA", "TEX",
        "PHI", "NYN", "SDN", "TOR", "SLN", "MIL", "MIN", "DET", "COL", "ARI",
        "MIA", "TBA", "KCA", "PIT", "CIN", "BAL", "WAS", "ATH", "ANA", "CHA",
    ],
    "capacity": [
        34830, 46537, 56000, 41649, 37755, 41084, 41168, 41915, 47929, 40300,
        42792, 41922, 40209, 49282, 45494, 41900, 38544, 41083, 50144, 48633,
        36742, 25025, 37903, 38747, 42319, 45971, 41339, 46847, 45517, 40615,
    ],
})

mlb_home_games = (
    mlb_home_games
    .merge(
        payroll_df.rename(columns={"payroll": "team_payroll"}),
        on=["season", "team_abbr"],
        how="left",
    )
    .merge(
        payroll_df.rename(
            columns={"team_abbr": "opponent_abbr", "payroll": "opponent_payroll"}
        ),
        on=["season", "opponent_abbr"],
        how="left",
    )
    .merge(stadium_capacity, on="team_abbr", how="left")
)

mlb_home_games["attendance_pct"] = mlb_home_games["attendance"] / mlb_home_games["capacity"]
mlb_home_games["attendance_pct"] = mlb_home_games["attendance_pct"].clip(upper=1.20)
mlb_home_games["opponent_payroll_tier"] = pd.qcut(
    mlb_home_games["opponent_payroll"].rank(method="first"),
    q=4,
    labels=TIER_ORDER,
    duplicates="drop",
)

guardians_home_games = mlb_home_games[mlb_home_games["team_abbr"] == "CLE"].copy()

team_summary = (
    mlb_home_games
    .groupby(["team_abbr", "team", "season"], as_index=False)
    .agg(
        avg_attendance=("attendance", "mean"),
        avg_attendance_pct=("attendance_pct", "mean"),
        team_payroll=("team_payroll", "mean"),
        wins=("result", lambda s: (s == "W").sum()),
        losses=("result", lambda s: (s == "L").sum()),
    )
)

team_summary["games"] = team_summary["wins"] + team_summary["losses"]
team_summary["win_pct"] = team_summary["wins"] / team_summary["games"]
team_summary["is_guardians"] = team_summary["team_abbr"].eq("CLE")

mlb_home_games.to_csv("mlb_home_games_clean.csv", index=False)
guardians_home_games.to_csv("guardians_home_games_clean.csv", index=False)
team_summary.to_csv("team_summary_clean.csv", index=False)

display(team_summary.head())

,team_abbr,team,season,avg_attendance,avg_attendance_pct,team_payroll,wins,losses,games,win_pct,is_guardians
0,ANA,Los Angeles Angels,2021,18484.012195,0.406090,196428336.0,40,42,82,0.487805,False
1,ANA,Los Angeles Angels,2022,30339.024691,0.666543,191581966.0,40,41,81,0.493827,False
2,ANA,Los Angeles Angels,2023,32599.691358,0.716209,231476965.0,38,43,81,0.469136,False
3,ANA,Los Angeles Angels,2024,31822.185185,0.699127,193150799.0,32,49,81,0.395062,False
4,ANA,Los Angeles Angels,2025,32290.197531,0.709410,230062125.0,39,42,81,0.481481,False


## 4. Attendance by Month and Weekday

This heatmap shows how Guardians attendance changes by timing. Using capacity percentage makes the chart easier to compare than raw attendance.

In [21]:
pivot_pct = (
    guardians_home_games
    .pivot_table(values="attendance_pct", index="month", columns="weekday", aggfunc="mean")
    .reindex(index=MONTH_ORDER, columns=WEEKDAY_ORDER)
)

fig = px.imshow(
    pivot_pct,
    text_auto=False,
    aspect="auto",
    color_continuous_scale="RdYlGn",
    title="Guardians Stadium Utilization by Month and Weekday",
    labels={"x": "Weekday", "y": "Month", "color": "Capacity Used"},
)

fig.update_traces(
    text=pivot_pct.applymap(lambda value: "" if pd.isna(value) else f"{value:.0%}"),
    texttemplate="%{text}",
    hovertemplate="<b>%{y}</b><br>%{x}<br>Capacity used: %{z:.1%}<extra></extra>",
    xgap=3,
    ygap=3,
)

fig.update_layout(
    template="plotly_white",
    width=1000,
    height=520,
    font={"size": 14, "color": TEXT_COLOR},
    title={"x": 0.5, "xanchor": "center"},
    margin={"t": 80, "r": 80, "b": 70, "l": 90},
)
fig.update_xaxes(side="bottom")
fig.update_yaxes(autorange="reversed")
fig.show()

/tmp/ipykernel_9367/2197828516.py:17: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



## 5. Opponent Payroll Tier

This chart compares Guardians attendance percentage by opponent payroll tier and places team logos in the correct tier. Opponent teams are deduplicated so each logo appears once per tier.

In [22]:
ESPN_LOGO_ABBR = {
    "ARI": "ari", "ATL": "atl", "BAL": "bal", "BOS": "bos", "CHN": "chc",
    "CHA": "chw", "CIN": "cin", "CLE": "cle", "COL": "col", "DET": "det",
    "HOU": "hou", "KCA": "kc", "ANA": "laa", "LAN": "lad", "MIA": "mia",
    "MIL": "mil", "MIN": "min", "NYN": "nym", "NYA": "nyy", "ATH": "ath",
    "PHI": "phi", "PIT": "pit", "SDN": "sd", "SFN": "sf", "SEA": "sea",
    "SLN": "stl", "TBA": "tb", "TEX": "tex", "TOR": "tor", "WAS": "wsh",
}


def logo_url(team_abbr):
    return f"https://a.espncdn.com/i/teamlogos/mlb/500/{ESPN_LOGO_ABBR.get(team_abbr, team_abbr.lower())}.png"


payroll_summary = (
    guardians_home_games
    .groupby("opponent_payroll_tier", observed=False)["attendance_pct"]
    .mean()
    .reindex(TIER_ORDER)
    .reset_index()
)

teams_by_tier = (
    guardians_home_games
    .dropna(subset=["opponent_payroll_tier", "opponent_abbr"])
    .assign(opponent_abbr=lambda df: df["opponent_abbr"].astype(str).str.upper().str.strip())
    .drop_duplicates(["opponent_payroll_tier", "opponent_abbr"])
    .groupby("opponent_payroll_tier", observed=False)["opponent_abbr"]
    .apply(lambda teams: sorted(teams.unique()))
    .reindex(TIER_ORDER)
)

fig = px.bar(
    payroll_summary,
    x="opponent_payroll_tier",
    y="attendance_pct",
    text="attendance_pct",
    category_orders={"opponent_payroll_tier": TIER_ORDER},
    title="Guardians Stadium Utilization by Opponent Payroll Tier",
)

fig.update_traces(
    marker_color="#5964F2",
    texttemplate="%{text:.1%}",
    textposition="outside",
)

fig.update_layout(
    template="plotly_white",
    width=1000,
    height=560,
    font={"size": 15, "color": TEXT_COLOR},
    title={"x": 0.5, "xanchor": "center"},
    xaxis_title="Opponent Payroll Tier",
    yaxis_title="Attendance (% of Capacity)",
    margin={"t": 90, "r": 70, "b": 70, "l": 90},
)
fig.update_yaxes(tickformat=".0%", range=[0, payroll_summary["attendance_pct"].max() + 0.16])

logo_size_x = 0.15
logo_size_y = 0.075

for tier in TIER_ORDER:
    teams = teams_by_tier.loc[tier]
    if not isinstance(teams, list) or not teams:
        continue

    bar_height = payroll_summary.loc[
        payroll_summary["opponent_payroll_tier"].eq(tier),
        "attendance_pct",
    ].iloc[0]

    cols = min(4, len(teams))
    x_offsets = {
        1: [0],
        2: [-32, 32],
        3: [-48, 0, 48],
        4: [-60, -20, 20, 60],
    }[cols]

    top_y = min(bar_height - 0.08, 0.66)
    y_gap = 0.085

    for index, team in enumerate(teams):
        row = index // cols
        col = index % cols

        # Convert categorical tier to numerical index and add scaled pixel offset
        x_pos = TIER_ORDER.index(tier) + (x_offsets[col] / 150) # Use a scaling factor, e.g., 150 pixels per data unit

        fig.add_layout_image(
            {
                "source": logo_url(team),
                "xref": "x",
                "yref": "y",
                "x": x_pos,
                "y": top_y - row * y_gap,
                "sizex": logo_size_x,
                "sizey": logo_size_y,
                "xanchor": "center",
                "yanchor": "middle",
                "layer": "above",
            }
        )

fig.show()

## 6. Guardians vs. Comparable Teams

This line chart compares the Guardians with the Brewers and Mariners, two smaller-market teams that are useful benchmarks for attendance utilization.

In [26]:
comparison_teams_abbr = ["MIL", "SEA"]
comparison_team_names = {
    "MIL": "Milwaukee Brewers",
    "SEA": "Seattle Mariners",
}

monthly_compare = (
    mlb_home_games[mlb_home_games["team_abbr"].isin(["CLE"] + comparison_teams_abbr)]
    .copy()
)
monthly_compare["month"] = pd.Categorical(
    monthly_compare["month"],
    categories=MONTH_ORDER,
    ordered=True,
)

monthly_plot = (
    monthly_compare
    .groupby(["team", "month"], observed=False, as_index=False)["attendance_pct"]
    .mean()
    .assign(
        team_label=lambda df: df["team"].map(
            {
                "Cleveland Guardians": "Guardians",
                comparison_team_names["MIL"]: "Brewers",
                comparison_team_names["SEA"]: "Mariners",
            }
        )
    )
    .dropna(subset=["team_label"])
)

team_colors = {
    "Guardians": GUARDIANS_BLUE,
    "Brewers": "#FFC52F",
    "Mariners": "#005C5C",
}

dome_teams = {
    "Brewers": True,
    "Mariners": True,
}

fig = px.line(
    monthly_plot,
    x="month",
    y="attendance_pct",
    color="team_label",
    markers=True,
    category_orders={"month": MONTH_ORDER},
    color_discrete_map=team_colors,
    title="Monthly Home Attendance Utilization",
)

fig.update_traces(
    line={"width": 4},
    marker={"size": 10, "line": {"width": 2, "color": "white"}},
)

last_points = (
    monthly_plot
    .sort_values("month", key=lambda s: s.map({month: i for i, month in enumerate(MONTH_ORDER)}))
    .groupby("team_label")
    .tail(1)
)

for _, row in last_points.iterrows():
    fig.add_annotation(
        x=row["month"],
        y=row["attendance_pct"],
        text=f"{row['team_label']} {row['attendance_pct']:.1%}",
        showarrow=False,
        xshift=55,
        font={"size": 13, "color": team_colors[row["team_label"]]},
    )

    if dome_teams.get(row["team_label"]):
        fig.add_annotation(
            x=row["month"],
            y=row["attendance_pct"],
            text="(Dome)",
            showarrow=False,
            xshift=55,
            yshift=-18,
            font={"size": 11, "color": team_colors[row["team_label"]]},
        )

fig.update_layout(
    template="plotly_white",
    width=950,
    height=560,
    font={"size": 15, "color": TEXT_COLOR},
    title={
        "text": "Monthly Home Attendance Utilization<br><sup>Guardians vs. Brewers vs. Mariners</sup>",
        "x": 0.5,
        "xanchor": "center",
    },
    xaxis_title="Month",
    yaxis_title="Attendance (% of Capacity)",
    showlegend=False,
    margin={"t": 110, "r": 115, "b": 70, "l": 85},
    hovermode="x unified",
)
fig.update_yaxes(tickformat=".0%", range=[0.40, 0.85], gridcolor=GRID_COLOR)
fig.update_xaxes(gridcolor="#EEF2F7")
fig.show()

## 7. Winning and Attendance

This scatterplot shows whether teams with higher win percentages also tend to have higher attendance utilization. Guardians seasons are highlighted in blue.

In [24]:
plot_df = team_summary.copy()

non_guardians_df = plot_df[~plot_df["is_guardians"]].dropna(
    subset=["win_pct", "avg_attendance_pct"]
)
guardians_df = plot_df[plot_df["is_guardians"]].dropna(
    subset=["win_pct", "avg_attendance_pct"]
)

fig = px.scatter(
    non_guardians_df,
    x="win_pct",
    y="avg_attendance_pct",
    color_discrete_sequence=["#8A8F98"],
    hover_name="team",
    hover_data={
        "season": True,
        "wins": True,
        "losses": True,
        "avg_attendance": ":,.0f",
        "win_pct": ":.1%",
        "avg_attendance_pct": ":.1%",
    },
    title="Winning and Home Attendance",
)

fig.update_traces(
    name="Other MLB Teams",
    marker={"size": 9, "color": "#8A8F98", "line": {"width": 0.5, "color": "white"}},
    opacity=0.6,
    showlegend=True,
)

fig.add_trace(
    go.Scatter(
        x=guardians_df["win_pct"],
        y=guardians_df["avg_attendance_pct"],
        mode="markers",
        name="Guardians",
        marker={
            "size": 15,
            "color": GUARDIANS_BLUE,
            "symbol": "star",
            "line": {"width": 1.5, "color": "white"},
        },
        text=guardians_df["team"],
        customdata=guardians_df[["season", "wins", "losses", "avg_attendance"]],
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Season: %{customdata[0]}<br>"
            "Wins: %{customdata[1]}<br>"
            "Losses: %{customdata[2]}<br>"
            "Average attendance: %{customdata[3]:,.0f}<br>"
            "Win percentage: %{x:.1%}<br>"
            "Attendance percentage: %{y:.1%}"
            "<extra></extra>"
        ),
    )
)

ols_model = smf.ols("avg_attendance_pct ~ win_pct", data=non_guardians_df).fit()
trendline_x = np.linspace(non_guardians_df["win_pct"].min(), non_guardians_df["win_pct"].max(), 100)
trendline_y = ols_model.predict(pd.DataFrame({"win_pct": trendline_x}))

fig.add_trace(
    go.Scatter(
        x=trendline_x,
        y=trendline_y,
        mode="lines",
        name="League Trend",
        line={"color": TEXT_COLOR, "width": 3, "dash": "dash"},
        hoverinfo="skip",
    )
)

for _, row in guardians_df.iterrows():
    fig.add_annotation(
        x=row["win_pct"],
        y=row["avg_attendance_pct"],
        text=str(row["season"]),
        showarrow=False,
        yshift=16,
        font={"size": 12, "color": GUARDIANS_BLUE},
    )

fig.update_layout(
    template="plotly_white",
    width=950,
    height=600,
    font={"size": 15, "color": TEXT_COLOR},
    title={
        "text": "Winning and Home Attendance<br><sup>Guardians seasons highlighted against other MLB teams</sup>",
        "x": 0.5,
        "xanchor": "center",
    },
    xaxis_title="Win Percentage",
    yaxis_title="Average Home Attendance (% of Capacity)",
    legend_title_text="",
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.03, "xanchor": "center", "x": 0.5},
    margin={"t": 120, "r": 70, "b": 70, "l": 90},
    hovermode="closest",
)
fig.update_xaxes(tickformat=".0%", gridcolor=GRID_COLOR)
fig.update_yaxes(tickformat=".0%", gridcolor=GRID_COLOR)
fig.show()

## 8. Payroll and Winning Efficiency

This chart connects payroll to win percentage and highlights whether teams overperform or underperform relative to expected wins from payroll.

In [25]:
plot_df = team_summary.copy()
plot_df["payroll_m"] = plot_df["team_payroll"] / 1_000_000

model_df = plot_df.dropna(subset=["payroll_m", "win_pct"]).copy()
ols_model = smf.ols("win_pct ~ payroll_m", data=model_df).fit()

plot_df["expected_win_pct"] = ols_model.predict(plot_df[["payroll_m"]])
plot_df["win_pct_vs_expected"] = plot_df["win_pct"] - plot_df["expected_win_pct"]
plot_df["performance"] = np.select(
    [
        plot_df["win_pct_vs_expected"] >= 0.05,
        plot_df["win_pct_vs_expected"] <= -0.05,
    ],
    ["Overperformed payroll", "Underperformed payroll"],
    default="Near expectation",
)

non_guardians = plot_df[~plot_df["is_guardians"]].copy()
guardians = plot_df[plot_df["is_guardians"]].copy()

fig = px.scatter(
    non_guardians,
    x="payroll_m",
    y="win_pct",
    color="performance",
    color_discrete_map={
        "Overperformed payroll": "#1A936F",
        "Near expectation": "#8A8F98",
        "Underperformed payroll": "#D1495B",
    },
    hover_name="team",
    hover_data={
        "season": True,
        "wins": True,
        "losses": True,
        "payroll_m": ":.1f",
        "win_pct": ":.1%",
        "expected_win_pct": ":.1%",
        "win_pct_vs_expected": ":.1%",
        "performance": False,
    },
    title="Payroll Buys Wins, But Efficiency Varies",
)

fig.update_traces(
    marker={"size": 10, "line": {"width": 0.6, "color": "white"}},
    opacity=0.65,
)

fig.add_trace(
    go.Scatter(
        x=guardians["payroll_m"],
        y=guardians["win_pct"],
        mode="markers",
        name="Guardians",
        marker={
            "symbol": "star",
            "size": 18,
            "color": GUARDIANS_BLUE,
            "line": {"width": 1.5, "color": "white"},
        },
        customdata=guardians[
            ["season", "wins", "losses", "payroll_m", "expected_win_pct", "win_pct_vs_expected"]
        ],
        text=guardians["team"],
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Season: %{customdata[0]}<br>"
            "Wins: %{customdata[1]}<br>"
            "Losses: %{customdata[2]}<br>"
            "Payroll: $%{customdata[3]:.1f}M<br>"
            "Win pct: %{y:.1%}<br>"
            "Expected win pct: %{customdata[4]:.1%}<br>"
            "Vs expected: %{customdata[5]:+.1%}"
            "<extra></extra>"
        ),
    )
)

x_line = np.linspace(model_df["payroll_m"].min(), model_df["payroll_m"].max(), 100)
y_line = ols_model.predict(pd.DataFrame({"payroll_m": x_line}))

fig.add_trace(
    go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        name="Expected win % from payroll",
        line={"color": TEXT_COLOR, "width": 3, "dash": "dash"},
        hoverinfo="skip",
    )
)

for _, row in guardians.iterrows():
    fig.add_annotation(
        x=row["payroll_m"],
        y=row["win_pct"],
        text=str(row["season"]),
        showarrow=False,
        yshift=18,
        font={"size": 13, "color": GUARDIANS_BLUE},
    )

fig.update_layout(
    template="plotly_white",
    width=980,
    height=640,
    font={"size": 15, "color": TEXT_COLOR},
    title={
        "text": "Payroll Buys Wins, But Efficiency Varies<br><sup>Guardians seasons highlighted against MLB team-seasons</sup>",
        "x": 0.5,
        "xanchor": "center",
    },
    xaxis_title="Team Payroll, USD Millions",
    yaxis_title="Win Percentage",
    legend_title_text="",
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.03,
        "xanchor": "center",
        "x": 0.5,
        "bgcolor": "rgba(255,255,255,0.85)",
    },
    margin={"t": 145, "r": 80, "b": 70, "l": 90},
    hovermode="closest",
)
fig.update_xaxes(tickprefix="$", ticksuffix="M", gridcolor=GRID_COLOR)
fig.update_yaxes(tickformat=".0%", gridcolor=GRID_COLOR)
fig.show()

## Recruiter Notes

### Key takeaways
- Attendance is strongest during summer months and weekend games.
- Higher-profile opponents are associated with stronger Guardians stadium utilization.
- Winning percentage is positively associated with attendance percentage across MLB team-seasons.
- Payroll is directionally associated with winning, but the gap between actual and expected performance shows why spending efficiency matters.

### Why this project matters
This project demonstrates end-to-end analytics work: sourcing public data, cleaning raw files, standardizing join keys, engineering features, and translating analysis into business recommendations.